In [ ]:
# Install dependencies
!pip install fairlearn aif360 kagglehub reportlab tensorflow matplotlib seaborn scikit-learn --quiet

import os
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models, applications, optimizers
from sklearn.model_selection import train_test_split
from fairlearn.metrics import MetricFrame, selection_rate, false_positive_rate
from fairlearn.postprocessing import ThresholdOptimizer
from fairlearn.reductions import ExponentiatedGradient, DemographicParity
from sklearn.linear_model import LogisticRegression
from sklearn.utils import resample
from reportlab.platypus import SimpleDocTemplate, Paragraph, Image, Spacer
from reportlab.lib.styles import getSampleStyleSheet
import kagglehub
import matplotlib.image as mpimg
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, accuracy_score
from sklearn.preprocessing import label_binarize
import itertools

In [ ]:
raw_path = kagglehub.dataset_download("aibloy/fairface")
local_data_dir = "data/fairface"
if os.path.exists(local_data_dir):
    shutil.rmtree(local_data_dir)
os.makedirs(local_data_dir, exist_ok=True)
for f in os.listdir(raw_path):
    shutil.move(os.path.join(raw_path, f), local_data_dir)
TRAIN_IMG_DIR = os.path.join(local_data_dir, "FairFace", "train")
VAL_IMG_DIR   = os.path.join(local_data_dir, "FairFace", "val")
TRAIN_CSV     = os.path.join(local_data_dir, "FairFace", "train_labels.csv")
VAL_CSV       = os.path.join(local_data_dir, "FairFace", "val_labels.csv")

# Load CSVs
df_train = pd.read_csv(TRAIN_CSV)
df_val   = pd.read_csv(VAL_CSV)

In [ ]:
# EDA plots
plt.figure(figsize=(10,5))
sns.countplot(data=df_train, x="race", hue="gender")
plt.xticks(rotation=45)
plt.title("Race × Gender Distribution (Train)")
plt.tight_layout()
plt.savefig("eda_distribution.png")
plt.show()

In [ ]:
# Number of samples per race to display
samples_per_race = 5
races = df_train['race'].unique()

plt.figure(figsize=(samples_per_race*2, len(races)*2.5))

for i, race in enumerate(races):
    # Get sample files for this race
    race_files = df_train[df_train['race'] == race]['file'].sample(samples_per_race, random_state=42)

    for j, fname in enumerate(race_files):
        img_path = os.path.join(TRAIN_IMG_DIR, fname)
        img = mpimg.imread(img_path)

        plt.subplot(len(races), samples_per_race, i*samples_per_race + j + 1)
        plt.imshow(img)
        plt.axis('off')
        plt.title(race, fontsize=8)  # Add race label on top of image

plt.suptitle("Sample Images by Race", fontsize=16)
plt.tight_layout()
plt.savefig("sample_images_by_race.png")
plt.show()

In [ ]:
# Data Generators
IMG_SIZE = 224
BATCH_SIZE = 32
train_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255, horizontal_flip=True, validation_split=0.1
)

In [ ]:
# Fix file paths for train and val
df_train['file'] = df_train['file'].apply(lambda x: x.strip())
df_val['file']   = df_val['file'].apply(lambda x: x.strip())

# If CSV contains full path, keep only filename
df_train['file'] = df_train['file'].apply(lambda x: os.path.basename(x))
df_val['file']   = df_val['file'].apply(lambda x: os.path.basename(x))

# Verify sample
print(df_train['file'].head())

# Now load generators
train_gen = train_datagen.flow_from_dataframe(
    dataframe=df_train,
    directory=TRAIN_IMG_DIR,
    x_col="file",
    y_col="race",
    subset="training",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=True
)
val_gen = train_datagen.flow_from_dataframe(
    dataframe=df_train,
    directory=TRAIN_IMG_DIR,
    x_col="file",
    y_col="race",
    subset="validation",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)


#Base Model Approach

In [ ]:
base_model_race = applications.ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)
x = layers.GlobalAveragePooling2D()(base_model_race.output)
output = layers.Dense(7, activation='softmax')(x)
model_race = models.Model(inputs=base_model_race.input, outputs=output)

model_race.compile(
    optimizer=optimizers.Adam(1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# === Train ===
history_race = model_race.fit(
    train_gen,
    validation_data=val_gen,
    epochs=5
)

In [ ]:
plt.figure()
plt.plot(history_race.history['accuracy'], label='Train')
plt.plot(history_race.history['val_accuracy'], label='Val')
plt.title("Baseline Training Accuracy")
plt.legend()
plt.savefig("baseline_accuracy.png")
plt.show()

In [ ]:
# === Predictions ===
y_scores = model_race.predict(val_gen)
y_pred = np.argmax(y_scores, axis=1)
y_true = val_gen.classes
class_names = list(val_gen.class_indices.keys())

# === Classification Report ===
print("\nClassification Report (Race Classification):")
print(classification_report(y_true, y_pred, target_names=class_names))

# === Confusion Matrix ===
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.title("Confusion Matrix - Race Classification")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
plt.savefig("cm_race.png")
plt.show()

# === ROC & AUC (One-vs-Rest for multi-class) ===
y_true_bin = label_binarize(y_true, classes=range(len(class_names)))

plt.figure(figsize=(8,6))
for i in range(len(class_names)):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_scores[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, lw=2, label=f"{class_names[i]} (AUC = {roc_auc:.3f})")

plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves - Race Classification")
plt.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.savefig("roc_curves_race.png")
plt.show()

In [ ]:
# Convert predictions to DataFrame
val_df_race = pd.DataFrame({
    "y_true": y_true,
    "y_pred": y_pred,
    "race": [class_names[i] for i in y_true]
})

# Accuracy and selection rate work fine for multi-class
metric_frame = MetricFrame(
    metrics={
        "accuracy": accuracy_score,
        "selection_rate": selection_rate
    },
    y_true=val_df_race["y_true"],
    y_pred=val_df_race["y_pred"],
    sensitive_features=val_df_race["race"]
)

print("\n=== Fairness Metrics by Race ===")
print(metric_frame.by_group)

# Compute per-class FPR manually (One-vs-Rest)
y_true_bin = label_binarize(val_df_race["y_true"], classes=range(len(class_names)))
y_pred_bin = label_binarize(val_df_race["y_pred"], classes=range(len(class_names)))

fpr_per_class = {}
for i, race_name in enumerate(class_names):
    FP = np.sum((y_pred_bin[:, i] == 1) & (y_true_bin[:, i] == 0))
    TN = np.sum((y_pred_bin[:, i] == 0) & (y_true_bin[:, i] == 0))
    fpr = FP / (FP + TN) if (FP + TN) > 0 else 0
    fpr_per_class[race_name] = fpr

print("\nFalse Positive Rate per Race (OvR):")
for race_name, fpr in fpr_per_class.items():
    print(f"{race_name}: {fpr:.3f}")

# Disparate Impact Ratio (accuracy ratio between first two race groups)
groups = metric_frame.by_group.index.tolist()
if len(groups) >= 2:
    acc_group_a = metric_frame.by_group.loc[groups[0], "accuracy"]
    acc_group_b = metric_frame.by_group.loc[groups[1], "accuracy"]
    disparate_impact_ratio = acc_group_a / acc_group_b if acc_group_b != 0 else None
    print(f"\nDisparate Impact Ratio ({groups[0]} vs {groups[1]}): {disparate_impact_ratio:.3f}")

# Equal Opportunity Difference using FPR difference
if len(groups) >= 2:
    eq_diff = fpr_per_class[groups[0]] - fpr_per_class[groups[1]]
    print(f"Equal Opportunity Difference ({groups[0]} - {groups[1]}): {eq_diff:.3f}")

In [ ]:
def evaluate_model(y_true, y_scores, class_names, title_prefix, save_prefix):
    y_pred = np.argmax(y_scores, axis=1)

    # Classification Report
    print(f"\nClassification Report - {title_prefix}")
    print(classification_report(y_true, y_pred, target_names=class_names))

    # Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8,6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f"{title_prefix} - Confusion Matrix")
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.tight_layout()
    plt.savefig(f"{save_prefix}_cm.png")
    plt.show()

    # ROC–AUC curves (multi-class OvR)
    y_true_bin = label_binarize(y_true, classes=range(len(class_names)))
    plt.figure(figsize=(8,6))
    for i in range(len(class_names)):
        fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_scores[:, i])
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, lw=2, label=f"{class_names[i]} (AUC = {roc_auc:.3f})")
    plt.plot([0, 1], [0, 1], 'k--', lw=2)
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"{title_prefix} - ROC Curves")
    plt.legend(loc="lower right", fontsize=8)
    plt.tight_layout()
    plt.savefig(f"{save_prefix}_roc.png")
    plt.show()

    # Fairness Metrics
    val_df_eval = pd.DataFrame({
        "y_true": y_true,
        "y_pred": y_pred,
        "race": [class_names[i] for i in y_true]
    })
    metric_frame = MetricFrame(
        metrics={"accuracy": accuracy_score, "selection_rate": selection_rate, "fpr": false_positive_rate},
        y_true=val_df_eval["y_true"], y_pred=val_df_eval["y_pred"],
        sensitive_features=val_df_eval["race"]
    )
    print(f"\nFairness Metrics by Race - {title_prefix}")
    print(metric_frame.by_group)

    # Disparate Impact Ratio & Equal Opportunity Difference (first 2 groups for example)
    groups = metric_frame.by_group.index.tolist()
    if len(groups) >= 2:
        acc_ratio = metric_frame.by_group.loc[groups[0], "accuracy"] / metric_frame.by_group.loc[groups[1], "accuracy"]
        eq_diff = metric_frame.by_group.loc[groups[0], "fpr"] - metric_frame.by_group.loc[groups[1], "fpr"]
        print(f"Disparate Impact Ratio ({groups[0]} vs {groups[1]}): {acc_ratio:.3f}")
        print(f"Equal Opportunity Difference ({groups[0]} - {groups[1]}): {eq_diff:.3f}")

# Pre-processing Approach

In [ ]:
# Oversample underrepresented race groups
df_balanced = pd.concat([
    resample(group, replace=True, n_samples=df_train['race'].value_counts().max(), random_state=42)
    for _, group in df_train.groupby('race')
])

train_gen_bal = train_datagen.flow_from_dataframe(
    dataframe=df_balanced,
    directory=TRAIN_IMG_DIR,
    x_col="file",
    y_col="race",
    subset="training",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=True
)

val_gen_bal = train_datagen.flow_from_dataframe(
    dataframe=df_balanced,
    directory=TRAIN_IMG_DIR,
    x_col="file",
    y_col="race",
    subset="validation",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

# Clone and retrain model
model_pre = tf.keras.models.clone_model(model_race)
model_pre.compile(optimizer=optimizers.Adam(1e-4), loss='categorical_crossentropy', metrics=['accuracy'])

history_pre = model_pre.fit(train_gen_bal, validation_data=val_gen, epochs=3)

In [ ]:
# Evaluate
y_scores_pre = model_pre.predict(val_gen)
evaluate_model(val_gen.classes, y_scores_pre, list(val_gen.class_indices.keys()), "Pre-processing", "preproc")

# In-processing Approach

In [ ]:
# Extract features from ResNet50 base model
feature_extractor = tf.keras.Model(inputs=model_race.input, outputs=model_race.layers[-2].output)

X_train_feats = feature_extractor.predict(train_gen)
y_train_labels = np.argmax(train_gen.labels, axis=0) if train_gen.class_mode == "categorical" else train_gen.labels
sensitive_train = [list(train_gen.class_indices.keys())[int(l)] for l in train_gen.classes]

X_val_feats = feature_extractor.predict(val_gen)
y_val_labels = val_gen.classes
sensitive_val = [list(val_gen.class_indices.keys())[int(l)] for l in val_gen.classes]

# Train fairness-aware logistic regression
clf = LogisticRegression(max_iter=1000, multi_class="ovr")
mitigator = ExponentiatedGradient(clf, constraints=DemographicParity())
mitigator.fit(X_train_feats, y_train_labels, sensitive_features=sensitive_train)

# Predictions
y_pred_inproc = mitigator.predict(X_val_feats)
y_scores_inproc = np.zeros((len(y_pred_inproc), len(val_gen.class_indices)))  # placeholder for ROC
for i, pred in enumerate(y_pred_inproc):
    y_scores_inproc[i, int(pred)] = 1  # one-hot for ROC

evaluate_model(y_val_labels, y_scores_inproc, list(val_gen.class_indices.keys()), "In-processing", "inproc")


# Post-processing Approach

In [ ]:
threshold_optimizer = ThresholdOptimizer(
    estimator=model_race,
    constraints="demographic_parity",
    prefit=True
)

threshold_optimizer.fit(val_gen.classes, val_gen.classes, sensitive_features=[list(val_gen.class_indices.keys())[int(l)] for l in val_gen.classes])

# Predictions
y_pred_post = threshold_optimizer.predict(val_gen.classes)
y_scores_post = np.zeros((len(y_pred_post), len(val_gen.class_indices)))
for i, pred in enumerate(y_pred_post):
    y_scores_post[i, int(pred)] = 1

evaluate_model(val_gen.classes, y_scores_post, list(val_gen.class_indices.keys()), "Post-processing", "postproc")
